This notebook collects all CDCR-specific data for state facilities, for exposure and vulnerability indices. The data produced gets joined to climate hazards in a different file.

In [53]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [54]:
# Facilities data filtered from FEMA's HiFLD data
facilities = pd.read_csv("data/cdcr_facilities_base.csv")
len(facilities)

84

## Import data

In [55]:
cdcr_pop = pd.read_csv("data_sources/facilities/cdcr/CDCR_2025_pop_averages.csv")
len(cdcr_pop)

31

In [56]:
cdcr_age = pd.read_csv("data_sources/facilities/cdcr/cdcr_in-custody-age_2025.csv")
cdcr_countryofbirth = pd.read_csv("data_sources/facilities/cdcr/cdcr_in-custody-countryofbirth_2025.csv")
cdcr_gender = pd.read_csv("data_sources/facilities/cdcr/cdcr_in-custody-gender_2025.csv")
cdcr_race = pd.read_csv("data_sources/facilities/cdcr/cdcr_in-custody-race_2025.csv")

In [57]:
cdcr_metadata = pd.read_csv("data_sources/facilities/cdcr/cdcr_manual_data.csv")
len(cdcr_metadata)

31

## Get CDCR facility abbreviation code

In [58]:
def process_cdcr_name(df):
    """
    Processes the facility name to get CDCR facility attributes.

    - Extract CDCR codes from names (e.g., (KVSP)). Manually set Avenal State Prison (10000826 -> ASP).
    - Correct FEMA acronym mismatches: CCFW -> CCWF (letter transposition), FSP -> FOL (naming convention).
    - Identify fire camps.
    """
    
    df['CDCR_code'] = None
    df['CDCR_firecamp'] = False

    # FEMA name parentheticals that don't match CDCR's own codes
    fema_to_cdcr = {
        'CCFW': 'CCWF',  # Central California Women's Facility: FEMA has letters transposed
        'FSP': 'FOL',    # Folsom State Prison: FEMA uses FSP, CDCR uses FOL
    }

    def extract_info(row):
            
        name = str(row['name'])
        
        # Check for Fire Camp (Case insensitive search for 'CAMP')
        if 'CAMP' in name.upper():
            row['CDCR_firecamp'] = True
            
        # Extract Code in parenthesis: e.g., "Kern Valley (kvsp)"
        code_match = re.search(r'\(([^)]+)\)', name)
        if code_match:
            row['CDCR_code'] = code_match.group(1).strip().upper()
            # Remove the parenthesis and its content from the name
            name = re.sub(r'\s*\([^)]+\)', '', name).strip()

        # Correct FEMA acronyms that don't match CDCR's codes
        if row['CDCR_code'] in fema_to_cdcr:
            row['CDCR_code'] = fema_to_cdcr[row['CDCR_code']]

        # Manual override for Avenal State Prison (no parenthetical in FEMA name)
        if row['facilityid'] == 10000826:
            row['CDCR_code'] = 'ASP'
        
        row['name'] = name
        return row

    # Apply the extraction and cleaning logic row-wise
    df = df.apply(extract_info, axis=1)
    
    return df

In [59]:
facilities = process_cdcr_name(facilities)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Add Population and population attribute data

In [60]:
def merge_population_data(df, cdcr_pop):
    """
    Merges the 2025 population data into the facility GeoDataFrame.
    Uses 'cdcr_code' and 'Facility_Code' as the join keys.
    """

    cdcr_pop['Facility_Code'] = cdcr_pop['Facility_Code'].astype(str).str.strip()

    df = df.merge(
        cdcr_pop[['Facility_Code', 'Average_2025_Population']], 
        left_on='CDCR_code', 
        right_on='Facility_Code', 
        how='left'
    )

    if 'Facility_Code' in df.columns:
        df = df.drop(columns=['Facility_Code'])
    
    return df

In [61]:
facilities_pop = merge_population_data(facilities, cdcr_pop)

In [62]:
facilities_pop['capacity_percent_2025'] = np.where(
    facilities_pop['Average_2025_Population'].notnull(),
    facilities_pop['Average_2025_Population'] / facilities_pop['capacity'],
    np.nan
)

## Demographics

In [63]:
def clean_population_col(df):
    """
    Cleans the 'Population' column by:
    1. Removing commas.
    2. Converting '*', '**', and '(Blank)' to 0.
    3. Converting to numeric float.
    """
    df = df.copy()
    if 'Population' in df.columns:
        pop_series = df['Population'].astype(str).str.replace(',', '').str.strip()
        
        # Apply CDCR suppression rules and empty value logic
        # ** = < 50, using 25 as an estimate
        # * = < 10, using 5 as an estimate
        # (Blank) = 0
        pop_series = pop_series.replace('**', '25')
        pop_series = pop_series.replace('*', '5')
        pop_series = pop_series.replace(['(Blank)', 'nan', 'NaN', ''], '0')
        
        df['Population'] = pd.to_numeric(pop_series, errors='coerce').fillna(0)
    return df

In [64]:
def get_average_counts_df(age_df, cob_df, gender_df, race_df):
    """
    Produces a single DataFrame containing the average counts of all 
    variables across the four demographic topics, indexed by Location.
    """
    dfs = []
    
    # Process each topic into average counts per location and category
    for df, cat_col, prefix in [
        (age_df, 'Age Group', 'age'),
        (cob_df, 'Country of Birth', 'cob'),
        (gender_df, 'Gender', 'gender'),
        (race_df, 'Race', 'race')
    ]:
        temp = clean_population_col(df)
        # Average counts across time/months
        avg = temp.groupby(['Location', cat_col])['Population'].mean().reset_index()
        # Pivot to wide format
        pivot = avg.pivot(index='Location', columns=cat_col, values='Population').fillna(0)
        # Standardize column names
        pivot.columns = [f"{prefix}_{str(c).lower().replace(' ', '_').replace('/', '_').replace('-', '_')}" for c in pivot.columns]
        dfs.append(pivot)
        
    counts_df = pd.concat(dfs, axis=1).fillna(0)
    return counts_df

In [65]:
demographic_counts = get_average_counts_df(cdcr_age, cdcr_countryofbirth, cdcr_gender, cdcr_race)

In [66]:
def calculate_generic_percentages(facility_df, counts_df, prefix):
    """
    Calculates the average percentage of a demographic category per facility
    relative to the 'Average_2025_Population' column using pre-calculated counts.
    Used for Gender or other categories where no custom grouping is done.
    """
    category_cols = [c for c in counts_df.columns if c.startswith(f"{prefix}_")]
    pivot_df = counts_df[category_cols].copy()
    
    facility_df['CDCR_code'] = facility_df['CDCR_code'].astype(str).str.strip()
    pivot_df.index = pivot_df.index.astype(str).str.strip()
    
    merged_df = facility_df.merge(pivot_df, left_on='CDCR_code', right_index=True, how='left')
    
    for col in pivot_df.columns:
        pct_col_name = f"{col}_pct"
        ratios = np.where(
            merged_df['Average_2025_Population'] > 0,
            merged_df[col] / merged_df['Average_2025_Population'],
            0
        )
        # Round at .0001 and 1.0
        final_pacts = np.clip(ratios, 0, 1)
        merged_df[pct_col_name] = np.where(final_pacts < 0.001, 0, final_pacts)
            
    output_df = merged_df.drop(columns=list(pivot_df.columns))
    return output_df

In [67]:
def add_age_tier_demographics(df, counts_df):
    """
    Calculates age percentages for specific cumulative tiers using pre-calculated counts.
    """
    # Define group categories for aggregation
    def age_col(name): return f"age_{name.lower().replace(' ', '_').replace('/', '_').replace('-', '_')}"
    
    groups_65_plus = [age_col(g) for g in ['65-69 Years', '70-74 Years', '75-79 Years', '80-84 Years', '85-89 Years', '90-94 Years', '95 and Older']]
    groups_60_plus = [age_col('60-64 Years')] + groups_65_plus
    groups_55_plus = [age_col('55-59 Years')] + groups_60_plus
    groups_50_plus = [age_col('50-54 Years')] + groups_55_plus

    tiers = pd.DataFrame(index=counts_df.index)
    tiers['age_over_50_count'] = counts_df[counts_df.columns.intersection(groups_50_plus)].sum(axis=1)
    tiers['age_over_55_count'] = counts_df[counts_df.columns.intersection(groups_55_plus)].sum(axis=1)
    tiers['age_over_60_count'] = counts_df[counts_df.columns.intersection(groups_60_plus)].sum(axis=1)
    tiers['age_over_65_count'] = counts_df[counts_df.columns.intersection(groups_65_plus)].sum(axis=1)

    # Merge and calculate percentages
    df['CDCR_code'] = df['CDCR_code'].astype(str).str.strip()
    merged = df.merge(tiers, left_on='CDCR_code', right_index=True, how='left')

    for tier in ['50', '55', '60', '65']:
        count_col = f'age_over_{tier}_count'
        pct_col = f'age_over_{tier}_pct'
        if 'Average_2025_Population' in merged.columns:
            merged[pct_col] = np.where(merged['Average_2025_Population'] > 0, merged[count_col] / merged['Average_2025_Population'], 0)
        else:
            merged[pct_col] = 0
            
    return merged.drop(columns=['age_over_50_count', 'age_over_55_count', 'age_over_60_count', 'age_over_65_count'])

In [68]:
def calculate_race_percentages(facility_df, counts_df):
    """
    Specifically calculates percentages for race by aggregating into 
    white_pct and peopleofcolor_pct.
    """
    prefix = 'race'
    category_cols = [c for c in counts_df.columns if c.startswith(f"{prefix}_")]
    pivot_df = counts_df[category_cols].copy()
    
    facility_df['CDCR_code'] = facility_df['CDCR_code'].astype(str).str.strip()
    pivot_df.index = pivot_df.index.astype(str).str.strip()
    
    # Identify White vs POC
    white_col = f"{prefix}_white"
    other_cols = [c for c in category_cols if c != white_col]
    
    # Aggregate counts
    race_summary = pd.DataFrame(index=pivot_df.index)
    race_summary['white'] = pivot_df[white_col] if white_col in pivot_df.columns else 0
    race_summary['peopleofcolor'] = pivot_df[other_cols].sum(axis=1)

    merged_df = facility_df.merge(race_summary, left_on='CDCR_code', right_index=True, how='left')
    
    for col in ['white', 'peopleofcolor']:
        pct_col_name = f"race_{col}_pct"
        merged_df[pct_col_name] = np.where(
            merged_df['Average_2025_Population'] > 0,
            merged_df[col] / merged_df['Average_2025_Population'],
            0
        )
            
    output_df = merged_df.drop(columns=['white', 'peopleofcolor'])
    return output_df

In [69]:
f_pop_age = add_age_tier_demographics(facilities_pop, demographic_counts)

In [70]:
f_pop_gender = calculate_generic_percentages(f_pop_age, demographic_counts, 'gender')

In [71]:
f_pop_race = calculate_race_percentages(f_pop_gender, demographic_counts)

## Add manually reviewed misc facilities data

## Add housing cooling data (Reuters/CDCR)

In [72]:
def merge_cooling_data(df, cooling_path):
    """
    Merges housing cooling data from the Reuters/CDCR Excel file.

    Each row in the source is one HVAC unit in a housing building.
    Produces per-facility counts of buildings and units, and the
    percentage of each with each cooling type.

    For buildings with mixed cooling types (8 of 563 total), a building
    is counted under each type it has, so building percentages can sum > 100%.

    The cooling data uses FSP and SQRC where cdcr_facilities uses FOL and SQ
    (Folsom State Prison and San Quentin respectively).
    """
    cooling_to_cdcr = {
        'FSP': 'FOL',   # Folsom State Prison: Reuters uses FSP, CDCR pipeline uses FOL
        'SQRC': 'SQ',   # San Quentin Rehabilitation Center: Reuters uses SQRC, FEMA uses SQ
    }
    cooling_types = {
        'evaporation': 'Evaporation Cooling',
        'refrigeration': 'Refrigeration Cooling',
        'ventilation': 'Ventilation Without Cooling',
    }

    raw = pd.read_excel(cooling_path, sheet_name='CDCR_data_2025')
    raw['Site Acronym'] = raw['Site Acronym'].replace(cooling_to_cdcr)

    # Unit-level counts (each row = one HVAC unit)
    unit_counts = raw.groupby('Site Acronym').size().rename('n_housing_units')

    # Building-level counts (unique Building per site)
    buildings = raw.drop_duplicates(subset=['Site Acronym', 'Building'])
    building_counts = buildings.groupby('Site Acronym').size().rename('n_housing_buildings')

    summary = pd.DataFrame({'n_housing_buildings': building_counts, 'n_housing_units': unit_counts})

    for key, label in cooling_types.items():
        # Units: straightforward percentage
        unit_type = raw[raw['Type of Cooling'] == label].groupby('Site Acronym').size()
        summary[f'pct_units_{key}'] = (unit_type / unit_counts).fillna(0).round(4)

        # Buildings: count distinct buildings that have at least one unit of this type
        bldg_type = (raw[raw['Type of Cooling'] == label]
                     .drop_duplicates(subset=['Site Acronym', 'Building'])
                     .groupby('Site Acronym').size())
        summary[f'pct_buildings_{key}'] = (bldg_type / building_counts).fillna(0).round(4)

    summary = summary.reset_index().rename(columns={'Site Acronym': 'cdcr_code_cooling'})

    df = df.merge(summary, left_on='CDCR_code', right_on='cdcr_code_cooling', how='left')
    df = df.drop(columns=['cdcr_code_cooling'])

    return df

In [73]:
f_pop_cooling = merge_cooling_data(f_pop_race, "data_sources/facilities/cdcr/Reuters_CDCR_cooling.xlsx")

In [74]:
def merge_cdcr_metadata(gdf, cdcr_metadata):
    """
    Merges facility metadata (opening year, planned closure, etc.)
    """
    
    cdcr_metadata['Acronym'] = cdcr_metadata['Acronym'].astype(str).str.strip()

    gdf = gdf.merge(
        cdcr_metadata,
        left_on='CDCR_code',
        right_on='Acronym',
        how='left'
    )

    if 'Acronym' in gdf.columns:
        gdf = gdf.drop(columns=['Acronym'])

    # Lowercase new columns to maintain naming convention
    gdf.columns = [col.lower() for col in gdf.columns]

    return gdf

In [75]:
facilities_df = merge_cdcr_metadata(f_pop_cooling, cdcr_metadata)

## Add SB 601 Programs data

In [76]:
def merge_sb601_programs(df, sb601_path):
    """
    Adds two columns listing the programs each facility had operational capacity for
    during 2024-2025, based on the SB 601 dashboard data:
      - cognitive_behavioral_interventions: semicolon-separated list of CBI programs
      - rehabilitative_programs: semicolon-separated list of Rehabilitative Programs

    A program is included if its Operational Capacity was > 0 in any month of the year.
    Degree counts (e.g. Number of GED Certificates) are excluded because they have no
    Operational Capacity row.
    """
    sb601 = pd.read_csv(sb601_path, skiprows=1)  # skip the quarter-label row
    sb601.columns = ['institution', 'category', 'metric'] + list(sb601.columns[3:])

    # Keep only Operational Capacity rows
    capacity = sb601[sb601['metric'].str.endswith('- Operational Capacity')].copy()

    # Identify monthly columns before adding derived columns
    month_cols = capacity.columns[3:]  # Jul-24 through Apr-25

    # Convert monthly columns to numeric; treat blanks as 0
    for col in month_cols:
        capacity[col] = pd.to_numeric(capacity[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

    # Strip the " - Operational Capacity" suffix to get the program name
    capacity['program'] = capacity['metric'].str.replace(r'\s*-\s*Operational Capacity$', '', regex=True).str.strip()

    # A program "has capacity" if any monthly value > 0
    capacity = capacity[capacity[month_cols].max(axis=1) > 0]

    # Build semicolon-separated program lists per institution and category
    program_lists = (
        capacity.groupby(['institution', 'category'])['program']
        .apply(lambda progs: '; '.join(sorted(progs)))
        .reset_index()
    )

    col_map = {
        'Cognitive Behavioral Intervention': 'cognitive_behavioral_interventions',
        'Rehabilitative Programs': 'rehabilitative_programs',
    }

    for category_label, col_name in col_map.items():
        subset = program_lists[program_lists['category'] == category_label][['institution', 'program']]
        subset = subset.rename(columns={'institution': 'cdcr_code', 'program': col_name})
        df = df.merge(subset, on='cdcr_code', how='left')

    return df

In [77]:
facilities_df = merge_sb601_programs(
    facilities_df,
    "data_sources/facilities/CDCR/sb601_programs_2024-2025.csv"
)

## Add CCHCS Health Care Population data (2025 averages)

In [78]:
def merge_cchcs_ipc(df, ipc_path):
    """
    Adds 2025 average CCHCS health classification percentages per facility.

    Source: CCHCS Health Care Services Dashboard, cchcs.ca.gov/dashboard/
    Measures are expressed as % of total facility population (confirmed: the four
    risk tiers sum to 100% for every facility-month, meaning the denominator is
    the total endorsed population, not a subset of active patients).

    Columns added:
      cchcs_high_risk_p1_pct_2025   - % High Risk Priority 1
      cchcs_high_risk_p2_pct_2025   - % High Risk Priority 2
      cchcs_medium_risk_pct_2025    - % Medium Risk
      cchcs_low_risk_pct_2025       - % Low Risk
      cchcs_mental_health_eop_pct_2025       - % Mental Health EOP
      cchcs_dpp_pct_2025            - % Disability Placement Program (DPP) Patients
      cchcs_age_over_50_pct_2025    - % Patients 50 Years or Older
      cchcs_specialized_beds_2025   - Specialized Health Care Beds (count, 12-month avg)
    """
    ipc = pd.read_csv(ipc_path)
    ipc['val'] = pd.to_numeric(
        ipc['value'].astype(str).str.replace('%', '').str.strip(), errors='coerce'
    )

    # Remap CCHCS codes that differ from our cdcr_code convention
    ipc['institution'] = ipc['institution'].replace({'FSP': 'FOL'})

    measures = {
        'High Risk Priority 1':                        'cchcs_high_risk_p1_pct_2025',
        'High Risk Priority 2':                        'cchcs_high_risk_p2_pct_2025',
        'Medium Risk':                                 'cchcs_medium_risk_pct_2025',
        'Low Risk':                                    'cchcs_low_risk_pct_2025',
        'Mental Health EOP':                           'cchcs_mental_health_eop_pct_2025',
        'Disability Placement Program (DPP) Patients': 'cchcs_dpp_pct_2025',
        'Patients 50 Years or Older':                  'cchcs_age_over_50_pct_2025',
        'Specialized Health Care Beds':                'cchcs_specialized_beds_2025',
    }

    ipc_2025 = ipc[ipc['month'].str.endswith('2025') & ipc['measure'].isin(measures)]

    for measure, col_name in measures.items():
        avg = (
            ipc_2025[ipc_2025['measure'] == measure]
            .groupby('institution')['val']
            .mean()
            .round(1)
            .reset_index()
            .rename(columns={'institution': 'cdcr_code', 'val': col_name})
        )
        df = df.merge(avg, on='cdcr_code', how='left')

    return df


In [79]:
facilities_df = merge_cchcs_ipc(
    facilities_df,
    "data_sources/facilities/CDCR/cchcs_ipc_2017-2025.csv"
)

## Add SCO Staffing data (2025 averages)

In [ ]:
def merge_sco_staffing(df, staffing_path):
    """
    Adds 2025 average SCO staff headcount per facility.

    sco_state_staff_2025:       CDCR operational + CCHCS healthcare employees
                                (non-PIA rows). For CHCF, the CCHCS row is summed
                                with the CDCR operational row.
    sco_incarcerated_staff_2025: Prison Industry Authority workers (PIA rows),
                                who are incarcerated people employed through PIA.
    """
    sco = pd.read_csv(staffing_path)

    state = (
        sco[~sco['is_pia']].groupby('cdcr_code')['total']
        .sum().round(1).reset_index()
        .rename(columns={'total': 'sco_state_staff_2025'})
    )
    incarcerated = (
        sco[sco['is_pia']].groupby('cdcr_code')['total']
        .sum().round(1).reset_index()
        .rename(columns={'total': 'sco_incarcerated_staff_2025'})
    )
    df = df.merge(state, on='cdcr_code', how='left')
    df = df.merge(incarcerated, on='cdcr_code', how='left')
    return df


In [81]:
facilities_df = merge_sco_staffing(
    facilities_df,
    "data_sources/facilities/CDCR/sco_staffing_2025_avg.csv"
)


## Export

In [82]:
facilities_df.to_csv("data/cdcr_facilities.csv", index=False)

In [83]:
facilities_df[facilities_df["cdcr_code"] == "CCWF"]

,facilityid,name,address,city,state,zip,telephone,type,status,population,...,rehabilitative_programs,cchcs_high_risk_p1_pct_2025,cchcs_high_risk_p2_pct_2025,cchcs_medium_risk_pct_2025,cchcs_low_risk_pct_2025,cchcs_mental_health_eop_pct_2025,cchcs_dpp_pct_2025,cchcs_age_over_50_pct_2025,cchcs_specialized_beds_2025,sco_total_staff_2025
17,10000801,Central California Women'S Facility,23370 Road 22,Chowchilla,CA,93610,(559) 665-5531,STATE,OPEN,NaN,...,Academic Education; Career Technical Education...,9.4,11.6,59.9,19.2,5.0,14.1,20.8,38.0,1263.3
